# PyTorch 环境
## 环境依赖安装

根据项目根目录下的 `requirement.txt` 统一安装依赖。

In [ ]:
%pip install -r ../requirement.txt

## 基础导入与版本验证

In [1]:
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Version:", torch.version.cuda)
print("CUDA Available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

PyTorch Version: 2.14.0+cpu
CUDA Version: None
CUDA Available: False
GPU: CPU


## 加速器可用性检查

深度学习计算主要分为通用主机处理器（CPU）与专用硬件加速器（Accelerator）。PyTorch 支持多种主流加速硬件：

In [2]:
# 1. NVIDIA GPU (CUDA) 检查
cuda_available = torch.cuda.is_available()
print(f"CUDA 是否可用: {cuda_available}")

if cuda_available:
    print(f"  GPU 设备总数: {torch.cuda.device_count()}")
    print(f"  当前设备索引: {torch.cuda.current_device()}")
    print(f"  设备型号: {torch.cuda.get_device_name(0)}")

# 2. Apple Silicon (MPS: Metal Performance Shaders) 检查
# 适用于搭载 M1/M2/M3/M4 等芯片的 macOS 系统
mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
print(f"Apple MPS 是否可用: {mps_available}")

CUDA 是否可用: False
Apple MPS 是否可用: False


# 核心概念：CPU 与 Accelerator 的内存模型

时刻清楚你的代码和数据正在何处物理执行。

物理层面的内存隔离:

- Host Memory（主机系统内存，RAM）：CPU 寻址和操作的内存空间。Python 原生对象、NumPy 数组、默认创建的 Tensor 最初均存放于此。
- Device Memory（设备显存，VRAM）：独立显卡或加速器上的高带宽专用内存。加速器的计算核心（如 CUDA Core / Tensor Core）只能高速读写显存中的数据。
![img1](./src/chapter01.png)

> 计算硬性规则：PyTorch 不允许两个位于不同物理设备上的张量直接进行算术运算（如加法、矩阵乘法）。算子与参与运算的所有张量必须处于同一个设备上。

## 核心 API 体系

### torch.device(...)：定义目标计算设备

推荐在所有项目中采用设备自适应代码模版，实现跨平台（NVIDIA 显卡、MacBook、普通电脑）无缝迁移：

In [3]:
# 优雅的设备自适应模版
def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print(f"当前选定的执行设备为: {device}")

当前选定的执行设备为: cpu


### Tensor.device：查询张量物理位置

用于动态调试与校验数据所在硬件：

x = torch.tensor([1.0, 2.0, 3.0])
print("默认创建位置:", x.device)  # 输出通常为: cpu

### .to(...)：设备迁移机制

| 操作对象            | 调用方式               | 内存行为                                 | 是否原地修改（In-place） | 必须显式重新赋值？                      |
| --------------- | ------------------ | ------------------------------------ | ---------------- | ------------------------------ |
| `torch.Tensor`  | `x = x.to(device)` | 显存/内存拷贝，并返回新的 Tensor                 | 否（Out-of-place）  | **必须赋值**                       |
| `nn.Module`（模型） | `model.to(device)` | 遍历模型中的 `Parameter` 和 Buffer，并迁移到目标设备 | 是（In-place）      | 不必须；通常直接 `model.to(device)` 即可 |

In [4]:
# --- A. 张量迁移（必须重新赋值） ---
x = torch.tensor([1.0, 2.0, 3.0])
x = x.to(device)  # 如果仅写 x.to(device)，x 依然停留在原设备！

# --- B. 模型迁移（原地修改生效） ---
import torch.nn as nn
model = nn.Linear(3, 1)
model.to(device)  # 内部所有权重与偏置已被搬迁到 target device

Linear(in_features=3, out_features=1, bias=True)

# dtype 基础与类型系统

## 存储、数值范围与精度

PyTorch 提供了覆盖不同位宽与特性的完整数据类型体系：

### 浮点类型 (Floating-Point Types) —— 神经网络的核心

torch.float32 / torch.float (单精度 float)：

- 结构：1 位符号 + 8 位指数 + 23 位尾数。

- 地位：深度学习的标准默认类型。权重初始化、层运算、梯度回传大多默认使用 FP32。

torch.float16 / torch.half (半精度)：

- 结构：1 位符号 + 5 位指数 + 10 位尾数。

- 特点：动态范围较窄（易发生下溢 Underflow），但能大幅节省显存并提升计算速度。
特点：动态范围较窄（易发生下溢 Underflow），但能大幅节省显存并提升计算速度。

torch.bfloat16 (Brain Floating Point)：

- 结构：1 位符号 + 8 位指数 + 7 位尾数。

- 特点：拥有与 FP32 相同的动态数值范围，但牺牲了尾数精度；在大模型（LLM）混合精度训练中已成为主流首选。

torch.float64 / torch.double (双精度)：

- 特点：极高精度，但显存开销翻倍、计算速度慢，一般仅用于高精度科学计算或数值稳定性调试。

### 整数类型 (Integer Types) —— 离散索引与离散标签

torch.int64 / torch.long：

- 地位：PyTorch 中最重要的整数类型。所有的分类任务标签（Targets）、张量索引（Indices）、nn.Embedding 查找表的输入必须是该类型。

torch.int32 / torch.int：

- 特点：标准整型，节省内存，但在 PyTorch 算子索引中不可直接替代 int64。

torch.uint8：

- 范围：$0 \sim 255$。常见于图像读取（像素原始值）。

### 布尔类型 (Boolean) —— 逻辑控制

torch.bool：

- 仅包含 True 和 False，广泛用于掩码（Attention Mask、Dropout Mask、阈值过滤）。

In [5]:
# 查询数据类型：Tensor.dtype

x = torch.tensor([1, 2, 3])
print(x.dtype)  # 输出: torch.int64

y = torch.tensor([1.0, 2.0, 3.0])
print(y.dtype)  # 输出: torch.float32

torch.int64
torch.float32


In [6]:
# 显式创建时指定：dtype=...
# 在张量创建函数中显式约束类型是最佳工程实践：

a = torch.zeros((3, 3), dtype=torch.float32)
b = torch.ones((2, 4), dtype=torch.int64)
c = torch.tensor([1, 0, 1], dtype=torch.bool)

In [8]:
# 类型转换机制
# 方式 A：通用方法 .to(dtype=...)（最推荐，支持同时迁移设备和类型）

x = torch.tensor([1, 2, 3])  # 当前是 int64

# 仅转换类型
x_float = x.to(torch.float32)

# 同时转换设备与类型（生产环境标准写法）
x_gpu_float = x.to(device="cpu", dtype=torch.float32)

In [9]:
# 快捷方法（语义清晰、代码紧凑）
x = torch.tensor([1, 2, 3])

x_float = x.float()   # 等价于 .to(torch.float32)
x_long  = x.long()    # 等价于 .to(torch.int64)
x_half  = x.half()    # 等价于 .to(torch.float16)
x_bool  = x.bool()    # 非 0 转为 True，0 转为 False

> PyTorch 中的张量类型转换是 非原地操作（Out-of-place），必须重新赋值（如 x = x.float()）。